<a href="https://colab.research.google.com/github/natalie400/PS-Agents/blob/main/UCS_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Uniform Cost Search (UCS) Homework Assignment
Goal: Navigate a trailer from Nairobi to Kisumu with the minimum travel distance.

Group Study Reference notes:
- State: Current town
- Step cost: Road distance in km
- Frontier: Priority Queue (heapq sorted by path cost g)
- Goal Test: Checked when expanding a node from the queue to ensure optimality!
"""

import heapq
from itertools import count  # Used as a standard tie-breaker in python heapq

# --- Configuration & Problem Setup ---
START = "Nairobi"
GOAL = "Kisumu"

# Road network data (distances in km)
ROADS = [
    ("Nairobi", "Thika", 42),
    ("Thika", "Murang'a", 40),
    ("Murang'a", "Nyeri", 65),
    ("Nyeri", "Nyahururu", 70),
    ("Nairobi", "Naivasha", 92),
    ("Naivasha", "Nakuru", 64),
    ("Naivasha", "Nyahururu", 95),
    ("Nyahururu", "Nakuru", 90),
    ("Nairobi", "Narok", 140),
    ("Narok", "Bomet", 75),
    ("Nakuru", "Mau Summit", 56),
    ("Nakuru", "Eldoret", 155),
    ("Mau Summit", "Londiani", 24),
    ("Londiani", "Muhoroni", 75),
    ("Muhoroni", "Kisumu", 45),
    ("Londiani", "Kericho", 45),
    ("Bomet", "Kericho", 55),
    ("Kericho", "Kisumu", 100),
    ("Bomet", "Kisii", 70),
    ("Kisii", "Kisumu", 115),
    ("Eldoret", "Kakamega", 90),
    ("Kakamega", "Kisumu", 55),
]


def build_graph(roads):
    """Helper to parse roads into a standard undirected adjacency list"""
    graph = {}
    for a, b, km in roads:
        graph.setdefault(a, []).append((b, km))
        graph.setdefault(b, []).append((a, km))
    return graph


def show_frontier(frontier):
    """Helper function to display current frontier states in sorted order (cheapest first)"""
    items = sorted(frontier)
    return ", ".join(f"{town} ({g})" for g, _, town, _ in items)


def uniform_cost_search(graph, start, goal):
    # Tie-breaker counter to avoid comparing lists/town strings inside heapq tuple
    tie = count()

    # Frontier tuple: (cumulative_cost_g, order_added, current_town, path_taken)
    frontier = [(0, next(tie), start, [start])]

    # Keeps track of the minimum cost we've seen to reach any town
    best_g = {start: 0}
    explored = set()
    step = 0

    print(f"UCS from {start} to {goal}\n")
    print(f"{'Step':<5}{'Expand':<12}{'g (km)':<8}Frontier after expansion")
    print("-" * 78)

    while frontier:
        # Pop node with the smallest total path cost (g)
        g, _, town, path = heapq.heappop(frontier)

        # Standard graph search check: ignore if we already found a shorter path
        if town in explored:
            continue

        step += 1
        explored.add(town)

        # Optimal Goal Test happens upon EXPANSION
        if town == goal:
            print(f"{step:<5}{town:<12}{g:<8}GOAL REACHED")
            return path, g

        notes = []
        for neighbour, km in graph[town]:
            new_g = g + km
            if neighbour in explored:
                continue

            # If we discovered a better way to get to this neighbour
            if neighbour not in best_g or new_g < best_g[neighbour]:
                best_g[neighbour] = new_g
                heapq.heappush(
                    frontier, (new_g, next(tie), neighbour, path + [neighbour])
                )
            else:
                # Record rejected paths for the step-by-step trace output
                notes.append(f"{neighbour} {new_g} rejected (have {best_g[neighbour]})")

        # Print state trace of execution
        line = show_frontier(frontier)
        print(f"{step:<5}{town:<12}{g:<8}{line}")
        for n in notes:
            print(f"{'':<25}-> {n}")

    return None, float("inf")


def main():
    graph = build_graph(ROADS)
    path, cost = uniform_cost_search(graph, START, GOAL)

    print("\n" + "=" * 78)
    if path is None:
        print("No route found.")
        return

    print("OPTIMAL PATH")
    print(" -> ".join(path))
    print("\nLeg-by-leg cost:")
    total = 0
    for a, b in zip(path, path[1:]):
        km = next(d for n, d in graph[a] if n == b)
        total += km
        print(f"  {a:<12} -> {b:<12} {km:>4} km   (cumulative {total} km)")
    print(f"\nTotal path cost: {cost} km")


if __name__ == "__main__":
    main()

UCS from Nairobi to Kisumu

Step Expand      g (km)  Frontier after expansion
------------------------------------------------------------------------------
1    Nairobi     0       Thika (42), Naivasha (92), Narok (140)
2    Thika       42      Murang'a (82), Naivasha (92), Narok (140)
3    Murang'a    82      Naivasha (92), Narok (140), Nyeri (147)
4    Naivasha    92      Narok (140), Nyeri (147), Nakuru (156), Nyahururu (187)
5    Narok       140     Nyeri (147), Nakuru (156), Nyahururu (187), Bomet (215)
6    Nyeri       147     Nakuru (156), Nyahururu (187), Bomet (215)
                         -> Nyahururu 217 rejected (have 187)
7    Nakuru      156     Nyahururu (187), Mau Summit (212), Bomet (215), Eldoret (311)
                         -> Nyahururu 246 rejected (have 187)
8    Nyahururu   187     Mau Summit (212), Bomet (215), Eldoret (311)
9    Mau Summit  212     Bomet (215), Londiani (236), Eldoret (311)
10   Bomet       215     Londiani (236), Kericho (270), Kisii (285),